# servellm: serving Qwen2.5-1.5B on a T4 with vLLM and measuring continuous batching

Runs the five experiments from `CONCEPTS.md` section 9. Needs a GPU session (Kaggle: Settings → Accelerator → GPU T4 x2 or P100; Colab: T4).

Expected runtime: ~10 min install, ~5 min model load/warm-up, ~15 min experiments. Results are written to `results/` and zipped at the end for download.

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

## 1. Install
vLLM is pinned so results are reproducible. The wheel brings its own PyTorch (CUDA 12.9); the install takes several minutes on Kaggle.

In [ ]:
# Install notes (each line earned by a failed run, see results/kaggle_run*/README.md):
# - cu129 release wheel + matching PyTorch index (run 2: default wheel mixed CUDA versions with image packages)
# - remove the image's preinstalled torchaudio and torchcodec first: vLLM pins torchaudio==2.11.0 and
#   torchcodec>=0.14, and the image's copies (built for other CUDA versions) satisfied the pins, so pip kept
#   them and they crashed at import (runs 2 and 3). torchcodec's import guard catches ImportError but not the
#   OSError a bad .so raises, so it must be either matching or absent.
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
!pip install -q --no-deps --index-url https://download.pytorch.org/whl/cu129 "torchcodec>=0.14" || (pip uninstall -q -y torchcodec; echo "no cu129 torchcodec wheel; leaving it uninstalled (audio decoding not needed)")
# Clone under a name that is not the package name and install NON-editable: an editable
# install's .pth entry only takes effect on interpreter restart (run 1).
![ -d /kaggle/working/servellm-repo ] || git clone -q https://github.com/NomanJafar/servellm.git /kaggle/working/servellm-repo
%pip install -q "/kaggle/working/servellm-repo[plots]"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, servellm.launch
print("vllm", vllm.__version__, "| torch", torch.__version__, "| torch cuda", torch.version.cuda, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available: torch build vs driver mismatch, see nvidia-smi above"
!pip list 2>/dev/null | grep -i -E "^torch|^vllm|^transformers|^xformers|^flash"
# The exact import chain that killed run 3 (vllm serve -> multimodal -> media.audio -> torchcodec):
!python -c "import vllm.multimodal.media.audio as a; print('vllm multimodal import chain OK; torchcodec:', 'absent' if a._torchcodec_import_exc else 'present')"

## 2. Launch the server and read the startup log

Three numbers to look at: KV cache memory, KV cache size in tokens, and maximum concurrency. Compare against the hand calculation from CONCEPTS.md section 4 (≈28 KB per token for this model in FP16).

In [ ]:
from servellm.launch import ServerConfig, VllmServer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
cfg_on = ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, max_num_seqs=256)
server = VllmServer(cfg_on, log_path="results/startup_batching_on.log")
info = server.start(ready_timeout_s=900)
info

In [ ]:
BYTES_PER_TOKEN = 2 * 28 * 2 * 128 * 2   # K&V x layers x kv_heads x head_dim x fp16
if info.kv_cache_tokens and info.kv_cache_gib:
    predicted_gib = info.kv_cache_tokens * BYTES_PER_TOKEN / 2**30
    print(f"vLLM reports {info.kv_cache_gib:.2f} GiB for {info.kv_cache_tokens:,} tokens")
    print(f"hand calc  : {info.kv_cache_tokens:,} tokens x {BYTES_PER_TOKEN} B = {predicted_gib:.2f} GiB")
    print(f"max concurrent 4096-token requests: {info.kv_cache_tokens/4096:.1f} (vLLM says {info.max_concurrency}x)")
else:
    print("Log format changed; open results/startup_batching_on.log and update parse_startup_log patterns.")
    print(open("results/startup_batching_on.log").read()[-3000:])

## 3. Single-stream baseline and concurrency sweep (batching on)

Fixed 256-token prompt, exactly 128 output tokens per request (`ignore_eos`), two rounds per worker. Watch the printed line per level: throughput should climb steeply while ITL barely moves, then bend.

In [ ]:
!python -m servellm.sweep --model {MODEL} --out results/sweep_batching_on.json sweep --levels 1,2,4,8,16,32,64 --prompt-tokens 256 --max-tokens 128 --rounds 2

## 4. Mixed lengths: do short requests get stuck behind long ones?

8 requests wanting 16 tokens and 8 wanting 512 tokens, launched together. Under continuous batching the short ones should finish in well under a second while the long ones keep running.

In [ ]:
!python -m servellm.sweep --model {MODEL} --out results/mixed.json mixed --short 16 --long 512 --n-each 8

## 5. Batching off: `--max-num-seqs 1`

Same sweep with the batch capped at one sequence. The server now processes requests one after another; the gap between the two throughput curves is what continuous batching is worth on this hardware. Levels stop at 16 because each level takes concurrency x rounds x ~1.5 s serially.

In [ ]:
server.stop()
cfg_off = ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, max_num_seqs=1)
server = VllmServer(cfg_off, log_path="results/startup_batching_off.log")
server.start(ready_timeout_s=900)

In [ ]:
!python -m servellm.sweep --model {MODEL} --out results/sweep_batching_off.json sweep --levels 1,2,4,8,16 --prompt-tokens 256 --max-tokens 128 --rounds 2

In [ ]:
server.stop()

## 6. Plots and summary

In [ ]:
import json
from IPython.display import Image, display
from servellm.plots import plot_sweep, plot_mixed

on, off = json.load(open("results/sweep_batching_on.json")), json.load(open("results/sweep_batching_off.json"))
paths = plot_sweep({"batching on": on, "batching off (max-num-seqs=1)": off}, "results")
paths += plot_mixed(json.load(open("results/mixed.json")), "results")
for p in paths:
    display(Image(str(p)))

In [ ]:
print(f"{'conc':>5} {'tok/s on':>9} {'tok/s off':>9} {'ITL p50 on':>11} {'ITL p50 off':>12} {'TTFT p50 on':>12} {'TTFT p50 off':>13}")
off_by_c = {lv["concurrency"]: lv for lv in off["levels"]}
for lv in on["levels"]:
    o = off_by_c.get(lv["concurrency"])
    f = lambda d, k, s=1: f"{d[k]*s:.1f}" if d and d.get(k) is not None else "-"
    print(f"{lv['concurrency']:>5} {f(lv,'output_tokens_per_s'):>9} {f(o,'output_tokens_per_s'):>9} "
          f"{f(lv,'itl_p50_s',1000):>9}ms {f(o,'itl_p50_s',1000):>10}ms {f(lv,'ttft_p50_s',1000):>10}ms {f(o,'ttft_p50_s',1000):>11}ms")

## 7. Save
Download `results.zip`, unzip into the repo's `results/` folder, and commit. The README's Results section is written from these files.

In [ ]:
!cd results && zip -qr ../results.zip . && cd .. && ls -la results.zip && echo "GPU:" && nvidia-smi --query-gpu=name --format=csv,noheader